# Chapter 4 — Authoritative table reproduction

This notebook reproduces, in one deterministic pass, the Chapter 4 tables that
were recomputed on the final 70/10/20 split with the exact configuration of
`ml/05_train_validation_test.ipynb`. It supersedes the exploratory values saved
in the earlier notebooks (`01_evaluate_hurdle.ipynb`,
`02_evaluate_class_bins.ipynb`, `04_alsrs_model_process.ipynb`), whose outputs
predate the final split and configuration.

Tables produced here: 4.3, 4.4, 4.5, 4.6, 4.8, 4.10, 4.12 and 4.18.

Run from the `ml/` directory with the `agri_land_env` environment. Seed 42 and
the by-farm split make the results exactly reproducible.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

ML_DIR = Path.cwd() if Path.cwd().name == "ml" else Path.cwd() / "ml"

FEATURES = ["month", "biweek", "mean_C", "std_C", "precip_total_mm", "precip_rainy_days", "pet_mm",
            "spei_1m", "spei_3m", "spei_6m", "spei_12m", "AWC_mm", "Storage_mm", "P_acum_mm",
            "WRSI_1m", "deficit_1m", "oni"]
TARGETS = ["future_deficit_1m", "future_deficit_3m", "future_deficit_6m"]

REGRESSOR_CONFIG = dict(n_estimators=300, max_depth=None, min_samples_leaf=1, random_state=42, n_jobs=-1)
CLASSIFIER_CONFIG = dict(n_estimators=300, max_depth=None, min_samples_leaf=1, class_weight="balanced", random_state=42, n_jobs=-1)
PROB_THRESHOLD = 0.5

SCHEMES = {
    "4_classes": {"thresholds": [15, 30, 50], "names": ["LOW", "MEDIUM", "HIGH", "NOT_SUITABLE"]},
    "3_classes": {"thresholds": [15, 50], "names": ["LOW", "MODERATE", "SEVERE"]},
    "2_classes@20": {"thresholds": [20], "names": ["MODERATE", "SEVERE"]},
    "2_classes@30": {"thresholds": [30], "names": ["MODERATE", "SEVERE"]},
}


def weight_linear(d):
    return d


def weight_squared(d):
    return d ** 2


def weight_plus1sq(d):
    return 1 + d ** 2


def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, p)))


def bin_vector(v, thresholds):
    return np.searchsorted(thresholds, np.asarray(v), side="left")


def worst_deficit(preds):
    return np.maximum(np.maximum(np.asarray(preds[TARGETS[0]]),
                                 np.asarray(preds[TARGETS[1]])),
                      np.asarray(preds[TARGETS[2]]))


def recall(true_idx, pred_idx, k):
    m = true_idx == k
    return float((pred_idx[m] == k).mean()) if m.sum() else float("nan")


def classify4(v):
    if pd.isna(v):
        return "NA"
    if v <= 15:
        return "LOW"
    if v <= 30:
        return "MEDIUM"
    if v <= 50:
        return "HIGH"
    return "NOT_SUITABLE"


## Data and by-farm split

Cacao is loaded and split by farm into 70% train / 10% validation / 20% test,
identical to `ml/05`. All tables below are reported on the 10% validation set,
except the feature importances, which are computed on the 70% train set of each
crop (Section 4.18).


In [2]:
df = pd.read_csv(ML_DIR / "ml_dataset_cacao_ccn51.csv")
point_ids = df["point_id"].unique()
train_pts, rest = train_test_split(point_ids, test_size=0.30, random_state=42)
val_pts, test_pts = train_test_split(rest, test_size=2 / 3, random_state=42)
tr = df[df["point_id"].isin(train_pts)]
va = df[df["point_id"].isin(val_pts)]
Xtr, Xva = tr[FEATURES], va[FEATURES]
ytr = {t: tr[t].to_numpy() for t in TARGETS}
yva = {t: va[t].to_numpy() for t in TARGETS}
true_worst = worst_deficit(yva)

# Single random forest baseline (linear weighting) — Table 4.3.
baseline_preds = {}
baseline = {"mae": {}, "rmse": {}}
for t in TARGETS:
    m = RandomForestRegressor(**REGRESSOR_CONFIG)
    m.fit(Xtr, ytr[t], sample_weight=weight_linear(ytr[t]))
    p = m.predict(Xva)
    baseline_preds[t] = p
    baseline["mae"][t] = round(mean_absolute_error(yva[t], p), 3)
    baseline["rmse"][t] = round(rmse(yva[t], p), 3)

# Hurdle (classifier + regressor + gate) — Tables 4.5, 4.8, 4.10, 4.12.
expected_preds, gated_preds, hurdle_mae = {}, {}, {}
for t in TARGETS:
    clf = RandomForestClassifier(**CLASSIFIER_CONFIG)
    clf.fit(Xtr, (ytr[t] > 0).astype(int))
    pos = ytr[t] > 0
    reg = RandomForestRegressor(**REGRESSOR_CONFIG)
    reg.fit(Xtr[pos], ytr[t][pos], sample_weight=weight_linear(ytr[t][pos]))
    p = clf.predict_proba(Xva)[:, 1]
    mu = reg.predict(Xva)
    expected_preds[t] = p * mu
    gated_preds[t] = np.where(p >= PROB_THRESHOLD, mu, 0.0)
    hurdle_mae[t] = round(mean_absolute_error(yva[t], gated_preds[t]), 3)

gated_worst = worst_deficit(gated_preds)

print("Table 4.3 — baseline MAE/RMSE:", baseline)
print("Table 4.5 — hurdle (gated) MAE:", hurdle_mae)


Table 4.3 — baseline MAE/RMSE: {'mae': {'future_deficit_1m': 11.556, 'future_deficit_3m': 11.519, 'future_deficit_6m': 11.049}, 'rmse': {'future_deficit_1m': 13.986, 'future_deficit_3m': 13.212, 'future_deficit_6m': 13.332}}
Table 4.5 — hurdle (gated) MAE: {'future_deficit_1m': 3.988, 'future_deficit_3m': 4.587, 'future_deficit_6m': 7.241}


## Table 4.6 — Phantom deficit

Mean prediction on the rows where the true one-month deficit is zero. The
baseline emits about 11.6 points on those rows; the gated hurdle emits about 1.


In [3]:
T1 = "future_deficit_1m"
zero_mask = yva[T1] == 0
phantom = {
    "baseline": round(float(baseline_preds[T1][zero_mask].mean()), 2),
    "p_magnitude": round(float(expected_preds[T1][zero_mask].mean()), 2),
    "gated": round(float(gated_preds[T1][zero_mask].mean()), 2),
}
print("Table 4.6 — phantom:", phantom)


Table 4.6 — phantom: {'baseline': 11.56, 'p_magnitude': 1.74, 'gated': 1.07}


## Table 4.10 — Class-scheme accuracy

Accuracy of the same continuous gated predictions re-binned into 4, 3 and 2
classes. Fewer classes raise accuracy from 0.685 to 0.848–0.877.


In [4]:
scheme_acc = {}
for name, cfg in SCHEMES.items():
    true_idx = bin_vector(true_worst, cfg["thresholds"])
    pred_idx = bin_vector(gated_worst, cfg["thresholds"])
    scheme_acc[name] = round(accuracy_score(true_idx, pred_idx), 3)
print("Table 4.10 — class-scheme accuracy:", scheme_acc)


Table 4.10 — class-scheme accuracy: {'4_classes': 0.685, '3_classes': 0.768, '2_classes@20': 0.848, '2_classes@30': 0.877}


## Table 4.12 — Hurdle (gated) versus direct classifier

Three-class recall of the hurdle compared with a direct four-class
`RandomForestClassifier` merged into three classes.


In [5]:
true3 = bin_vector(true_worst, [15, 50])
hurdle3 = bin_vector(gated_worst, [15, 50])
hurdle_recall = {
    "LOW": round(recall(true3, hurdle3, 0), 3),
    "MODERATE": round(recall(true3, hurdle3, 1), 3),
    "SEVERE": round(recall(true3, hurdle3, 2), 3),
}

tr_worst = worst_deficit(ytr)
tr_sugg4 = pd.Series([classify4(v) for v in tr_worst], index=tr.index)
va_sugg4 = pd.Series([classify4(v) for v in true_worst], index=va.index)
clf4 = RandomForestClassifier(**CLASSIFIER_CONFIG)
clf4.fit(Xtr, tr_sugg4)
pred4 = pd.Series(clf4.predict(Xva), index=va.index)
merge3 = {"LOW": "LOW", "MEDIUM": "MODERATE", "HIGH": "MODERATE", "NOT_SUITABLE": "SEVERE"}
direct_recall = {}
for nm in ["LOW", "MODERATE", "SEVERE"]:
    true_m = va_sugg4.map(merge3) == nm
    pred_m = pred4.map(merge3) == nm
    direct_recall[nm] = round(float((pred_m[true_m]).mean()), 3) if true_m.sum() else None

print("Table 4.12 — hurdle recall:", hurdle_recall)
print("Table 4.12 — direct recall:", direct_recall)


Table 4.12 — hurdle recall: {'LOW': 0.746, 'MODERATE': 0.836, 'SEVERE': 0.73}
Table 4.12 — direct recall: {'LOW': 0.976, 'MODERATE': 0.375, 'SEVERE': 0.874}


## Table 4.4 — Weight comparison on the single regressor

MAE and three-class recall of the single regressor under three sample-weighting
schemes. Linear maximises SEVERE recall (0.722) and is adopted; its inflated
MAE is removed by the hurdle's first stage.


In [6]:
WEIGHTS = {"linear": weight_linear, "squared": weight_squared, "plus1sq": weight_plus1sq}
true_sugg = bin_vector(true_worst, [15, 50])
weight_rows = {}
for wname, wfn in WEIGHTS.items():
    preds, maes = {}, {}
    for t in TARGETS:
        m = RandomForestRegressor(**REGRESSOR_CONFIG)
        m.fit(Xtr, ytr[t], sample_weight=wfn(ytr[t]))
        preds[t] = m.predict(Xva)
        maes[t] = mean_absolute_error(yva[t], preds[t])
    sugg = bin_vector(worst_deficit(preds), [15, 50])
    weight_rows[wname] = {
        "mae_1m": round(maes["future_deficit_1m"], 3),
        "mae_3m": round(maes["future_deficit_3m"], 3),
        "mae_6m": round(maes["future_deficit_6m"], 3),
        "recall_MODERATE": round(recall(true_sugg, sugg, 1), 3),
        "recall_SEVERE": round(recall(true_sugg, sugg, 2), 3),
    }
print("Table 4.4 — weight comparison:", weight_rows)


Table 4.4 — weight comparison: {'linear': {'mae_1m': 11.556, 'mae_3m': 11.519, 'mae_6m': 11.049, 'recall_MODERATE': 0.905, 'recall_SEVERE': 0.722}, 'squared': {'mae_1m': 12.11, 'mae_3m': 13.142, 'mae_6m': 11.655, 'recall_MODERATE': 0.921, 'recall_SEVERE': 0.704}, 'plus1sq': {'mae_1m': 4.982, 'mae_3m': 6.737, 'mae_6m': 8.449, 'recall_MODERATE': 0.747, 'recall_SEVERE': 0.637}}


## Table 4.8 — Combination rules

Recall under three ways of combining the two hurdle stages. The gate restores
SEVERE recall (0.730) to the baseline level (0.722), unlike the naive product.


In [7]:
baseline_sugg = bin_vector(worst_deficit(baseline_preds), [15, 50])
expected_sugg = bin_vector(worst_deficit(expected_preds), [15, 50])
gated_sugg = bin_vector(worst_deficit(gated_preds), [15, 50])
comb_rows = {
    "baseline": {"recall_MODERATE": round(recall(true_sugg, baseline_sugg, 1), 3),
                 "recall_SEVERE": round(recall(true_sugg, baseline_sugg, 2), 3)},
    "p_magnitude": {"recall_MODERATE": round(recall(true_sugg, expected_sugg, 1), 3),
                    "recall_SEVERE": round(recall(true_sugg, expected_sugg, 2), 3)},
    "gate": {"recall_MODERATE": round(recall(true_sugg, gated_sugg, 1), 3),
             "recall_SEVERE": round(recall(true_sugg, gated_sugg, 2), 3)},
}
print("Table 4.8 — combination rules:", comb_rows)


Table 4.8 — combination rules: {'baseline': {'recall_MODERATE': 0.905, 'recall_SEVERE': 0.722}, 'p_magnitude': {'recall_MODERATE': 0.704, 'recall_SEVERE': 0.607}, 'gate': {'recall_MODERATE': 0.836, 'recall_SEVERE': 0.73}}


## Table 4.18 — Feature importance (stage 1 + stage 2)

Impurity-based importances, averaged over the classifier and the regressor and
across the three horizons, computed on the 70% train set of each crop. The
soil-water state dominates in both crops.


In [8]:
def importance(csv):
    d = pd.read_csv(ML_DIR / csv)
    pids = d["point_id"].unique()
    tpts, _ = train_test_split(pids, test_size=0.30, random_state=42)
    trr = d[d["point_id"].isin(tpts)]
    X = trr[FEATURES]
    imp = np.zeros(len(FEATURES))
    for t in TARGETS:
        y = trr[t].to_numpy()
        clf = RandomForestClassifier(**CLASSIFIER_CONFIG)
        clf.fit(X, (y > 0).astype(int))
        pos = y > 0
        reg = RandomForestRegressor(**REGRESSOR_CONFIG)
        reg.fit(X[pos], y[pos], sample_weight=weight_linear(y[pos]))
        imp += (clf.feature_importances_ + reg.feature_importances_) / 2
    imp /= len(TARGETS)
    return pd.Series(imp, index=FEATURES).sort_values(ascending=False)


feat_imp = {
    "cacao": importance("ml_dataset_cacao_ccn51.csv"),
    "sugarcane": importance("ml_dataset_sugarcane.csv"),
}
for crop, s in feat_imp.items():
    print(f"\n{crop} top-8:")
    for f, v in s.head(8).items():
        print(f"  {f:20s} {v:.4f}")



cacao top-8:
  P_acum_mm            0.1765
  Storage_mm           0.1348
  month                0.1161
  oni                  0.0665
  AWC_mm               0.0620
  mean_C               0.0592
  spei_1m              0.0554
  pet_mm               0.0435

sugarcane top-8:
  Storage_mm           0.1327
  AWC_mm               0.1165
  oni                  0.1086
  P_acum_mm            0.0971
  month                0.0797
  pet_mm               0.0714
  spei_1m              0.0572
  spei_12m             0.0494


In [ ]:
# Save a single summary artifact; the printed tables above are the authoritative
# values used in Chapter 4.
summary = {
    "baseline": baseline,
    "hurdle_mae": hurdle_mae,
    "phantom": phantom,
    "scheme_accuracy": scheme_acc,
    "hurdle_recall": hurdle_recall,
    "direct_recall": direct_recall,
    "weight_comparison": weight_rows,
    "combination_rules": comb_rows,
    "feature_importance": {c: {f: round(float(v), 4) for f, v in s.items()} for c, s in feat_imp.items()},
}
(ML_DIR / "ch4_tables.json").write_text(json.dumps(summary, indent=2))
print("Saved ch4_tables.json")
